# Revisión de la preparación de datos

Este notebook revisa los archivos generados por DVC. No crea ni modifica los datos oficiales.

## P1 — Tipado

**Pregunta:** ¿La primera transformación conserva todas las filas y convierte únicamente las fechas?

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data/raw/cfpb_reclamos_narrativa.parquet"
TYPED_PATH = PROJECT_ROOT / "data/interim/typed.parquet"

assert RAW_PATH.exists(), f"No se encontró {RAW_PATH}"
assert TYPED_PATH.exists(), f"No se encontró {TYPED_PATH}; ejecuta dvc repro"

### 1. Filas, columnas y tamaño

In [3]:
raw = pq.ParquetFile(RAW_PATH)
typed = pq.ParquetFile(TYPED_PATH)

file_summary = pd.DataFrame({
    "archivo": ["original", "tipado"],
    "filas": [raw.metadata.num_rows, typed.metadata.num_rows],
    "columnas": [raw.metadata.num_columns, typed.metadata.num_columns],
    "tamaño GiB": [RAW_PATH.stat().st_size / 1024**3, TYPED_PATH.stat().st_size / 1024**3],
})
file_summary["filas"] = file_summary["filas"].map("{:,}".format)
file_summary["tamaño GiB"] = file_summary["tamaño GiB"].map("{:.2f}".format)
display(file_summary)

assert typed.metadata.num_rows == raw.metadata.num_rows == 3_837_184
assert typed.metadata.num_columns == raw.metadata.num_columns == 16

,archivo,filas,columnas,tamaño GiB
0,original,"3,837,184",16,1.51
1,tipado,"3,837,184",16,0.85


### 2. Tipos antes y después

In [4]:
raw_schema = raw.schema_arrow
typed_schema = typed.schema_arrow
schema_comparison = pd.DataFrame({
    "columna": raw_schema.names,
    "tipo original": [str(field.type) for field in raw_schema],
    "tipo tipado": [str(field.type) for field in typed_schema],
})
schema_comparison["cambió"] = schema_comparison["tipo original"] != schema_comparison["tipo tipado"]
display(schema_comparison)

changed_columns = schema_comparison.loc[schema_comparison["cambió"], "columna"].tolist()
assert changed_columns == ["Date received", "Date sent to company"]
assert typed_schema.field("Date received").type == pa.date32()
assert typed_schema.field("Date sent to company").type == pa.date32()

,columna,tipo original,tipo tipado,cambió
0,Date received,large_string,date32[day],True
1,Product,large_string,large_string,False
2,Sub-product,large_string,large_string,False
3,Issue,large_string,large_string,False
4,Sub-issue,large_string,large_string,False
5,Consumer complaint narrative,large_string,large_string,False
6,Company public response,large_string,large_string,False
7,Company,large_string,large_string,False
8,State,large_string,large_string,False
9,ZIP code,large_string,large_string,False


### 3. Calidad de las fechas

In [5]:
typed_dates = pq.read_table(TYPED_PATH, columns=["Date received", "Date sent to company"])
date_summary = []
for column in typed_dates.column_names:
    values = typed_dates[column]
    date_summary.append({
        "columna": column,
        "mínimo": pc.min(values).as_py(),
        "máximo": pc.max(values).as_py(),
        "nulos": values.null_count,
    })
display(pd.DataFrame(date_summary))
assert all(row["nulos"] == 0 for row in date_summary)

,columna,mínimo,máximo,nulos
0,Date received,2015-03-19,2026-07-27,0
1,Date sent to company,2015-03-19,2026-08-13,0


### 4. Identificadores

In [6]:
complaint_ids = pq.read_table(TYPED_PATH, columns=["Complaint ID"])["Complaint ID"]
unique_ids = pc.count_distinct(complaint_ids).as_py()
display(pd.DataFrame({
    "medida": ["filas", "IDs únicos", "IDs nulos"],
    "valor": [len(complaint_ids), unique_ids, complaint_ids.null_count],
}))
assert unique_ids == len(complaint_ids) == 3_837_184
assert complaint_ids.null_count == 0

,medida,valor
0,filas,3837184
1,IDs únicos,3837184
2,IDs nulos,0


## Resultado de P1

- Se conservan las 3,837,184 filas y las 16 columnas.
- Solo cambian `Date received` y `Date sent to company`: pasan de texto a fecha.
- No se crean fechas nulas.
- Los 3,837,184 identificadores continúan presentes y son únicos.
- Ninguna categoría, narrativa u objetivo se modifica en esta etapa.

## P2 — Normalización de narrativas

**Normalizar** significa aplicar reglas fijas para que textos equivalentes puedan compararse. La versión `text_normalizer_v1` aplica Unicode NFKC, minúsculas con `casefold`, reemplazo de palabras formadas por dos o más `x` por `<redacted>`, unión de espacios y retiro de espacios al inicio y final.

Un **hash SHA-256** es un identificador estable calculado desde el texto normalizado. El mismo texto produce el mismo identificador; esto permite formar grupos sin usar la narrativa completa como llave.

In [7]:
from itertools import zip_longest

from src.data.normalize_text import (
    HASH_COLUMN,
    NARRATIVE_COLUMN,
    NORMALIZED_COLUMN,
    NORMALIZER_VERSION,
    hash_text,
    normalize_text,
)

NORMALIZED_PATH = PROJECT_ROOT / "data/interim/normalized.parquet"
assert NORMALIZED_PATH.exists(), f"No se encontró {NORMALIZED_PATH}; ejecuta dvc repro"

### 1. Filas, columnas y versión

In [8]:
normalized_file = pq.ParquetFile(NORMALIZED_PATH)
normalizer_version = normalized_file.schema_arrow.metadata[b"text_normalizer"].decode("utf-8")
p2_file_summary = pd.DataFrame({
    "archivo": ["tipado", "normalizado"],
    "filas": [typed.metadata.num_rows, normalized_file.metadata.num_rows],
    "columnas": [typed.metadata.num_columns, normalized_file.metadata.num_columns],
    "tamaño GiB": [TYPED_PATH.stat().st_size / 1024**3, NORMALIZED_PATH.stat().st_size / 1024**3],
})
p2_file_summary["filas"] = p2_file_summary["filas"].map("{:,}".format)
p2_file_summary["tamaño GiB"] = p2_file_summary["tamaño GiB"].map("{:.2f}".format)
display(p2_file_summary)
print(f"Versión: {normalizer_version}")

assert normalized_file.metadata.num_rows == typed.metadata.num_rows == 3_837_184
assert normalized_file.schema_arrow.names == typed.schema_arrow.names + [NORMALIZED_COLUMN, HASH_COLUMN]
assert normalizer_version == NORMALIZER_VERSION

,archivo,filas,columnas,tamaño GiB
0,tipado,"3,837,184",16,0.85
1,normalizado,"3,837,184",18,1.70


Versión: text_normalizer_v1


### 2. Conservación de la narrativa original

La comparación se realiza por lotes para revisar todas las filas sin cargar ambas tablas completas en memoria.

In [9]:
typed_batches = typed.iter_batches(batch_size=50_000, columns=[NARRATIVE_COLUMN])
normalized_batches = normalized_file.iter_batches(batch_size=50_000, columns=[NARRATIVE_COLUMN])
original_is_unchanged = True
for before, after in zip_longest(typed_batches, normalized_batches):
    if before is None or after is None or not before.equals(after):
        original_is_unchanged = False
        break

display(pd.DataFrame({"comprobación": ["narrativa original sin cambios"], "resultado": [original_is_unchanged]}))
assert original_is_unchanged

,comprobación,resultado
0,narrativa original sin cambios,True


### 3. Calidad de las columnas nuevas

In [10]:
def parquet_null_count(parquet_file, column_name):
    column_index = parquet_file.schema_arrow.get_field_index(column_name)
    return sum(
        parquet_file.metadata.row_group(index).column(column_index).statistics.null_count
        for index in range(parquet_file.metadata.num_row_groups)
    )

sample = next(normalized_file.iter_batches(batch_size=1_000, columns=[NORMALIZED_COLUMN, HASH_COLUMN]))
sample_normalized = sample.column(0).to_pylist()
sample_hashes = sample.column(1).to_pylist()
sample_hashes_are_correct = all(hash_text(text) == value for text, value in zip(sample_normalized, sample_hashes))
sample_is_idempotent = all(normalize_text(text) == text for text in sample_normalized)
normalized_nulls = parquet_null_count(normalized_file, NORMALIZED_COLUMN)
hash_nulls = parquet_null_count(normalized_file, HASH_COLUMN)

display(pd.DataFrame({
    "comprobación": ["textos normalizados nulos", "hashes nulos", "hash correcto en muestra", "regla estable en muestra"],
    "resultado": [normalized_nulls, hash_nulls, sample_hashes_are_correct, sample_is_idempotent],
}))
assert normalized_nulls == hash_nulls == 0
assert sample_hashes_are_correct
assert sample_is_idempotent

,comprobación,resultado
0,textos normalizados nulos,0
1,hashes nulos,0
2,hash correcto en muestra,True
3,regla estable en muestra,True


### 4. Grupos de texto normalizado

Un **grupo repetido** contiene al menos dos filas con el mismo hash y, por tanto, con el mismo texto después de aplicar las reglas aprobadas.

In [11]:
hashes = pq.read_table(NORMALIZED_PATH, columns=[HASH_COLUMN])[HASH_COLUMN].combine_chunks()
hash_counts = pc.value_counts(hashes)
counts = hash_counts.field("counts")
repeated_mask = pc.greater(counts, 1)
unique_hashes = len(counts)
repeated_groups = pc.sum(pc.cast(repeated_mask, pa.int64())).as_py()
rows_in_repeated_groups = pc.sum(pc.filter(counts, repeated_mask)).as_py()
repeated_percentage = rows_in_repeated_groups / len(hashes) * 100

display(pd.DataFrame({
    "medida": ["textos normalizados únicos", "grupos repetidos", "filas en grupos repetidos", "porcentaje de filas repetidas"],
    "valor": [f"{unique_hashes:,}", f"{repeated_groups:,}", f"{rows_in_repeated_groups:,}", f"{repeated_percentage:.2f}%"],
}))
assert unique_hashes == 2_476_614
assert repeated_groups == 281_943
assert rows_in_repeated_groups == 1_642_513

,medida,valor
0,textos normalizados únicos,"2,476,614"
1,grupos repetidos,"281,943"
2,filas en grupos repetidos,"1,642,513"
3,porcentaje de filas repetidas,42.81%


## Resultado de P2

- Se conservan las 3,837,184 filas y las 16 columnas originales.
- Se agregan el texto normalizado y su hash SHA-256; ninguna de estas columnas tiene valores ausentes.
- La regla aplicada queda identificada como `text_normalizer_v1`.
- Existen 2,476,614 textos normalizados distintos.
- 1,642,513 filas, 42.81%, comparten texto normalizado con otra fila.
- Compartir texto no demuestra que dos reclamos sean el mismo evento. El hash se usará para evitar que un mismo patrón textual quede dividido entre aprendizaje y evaluación.

## P3 — Taxonomía propuesta

Una **taxonomía** es una lista organizada de categorías. **Canonicalizar** significa asignar nombres históricos equivalentes a un nombre común. El mapa `taxonomy_v1_proposed` es explícito y revisable, pero continúa siendo una propuesta técnica sin revisión de negocio.

El mapa se congela con la información disponible hasta `2025-06-30`. Después del corte, una entrada nueva usa `__UNKNOWN__`; un motivo o una combinación producto–motivo no vistos se marcan como desconocidos y no actualizan el mapa.

In [12]:
from src.data.apply_taxonomy import (
    FREEZE_DATE,
    KNOWN_ISSUE_COLUMN,
    KNOWN_PAIR_COLUMN,
    KNOWN_PRODUCT_COLUMN,
    TAXONOMY_STATUS,
    TAXONOMY_VERSION,
)

TAXONOMY_PATH = PROJECT_ROOT / "data/interim/taxonomy.parquet"
PRODUCT_MAP_PATH = PROJECT_ROOT / "configs/product_taxonomy_v1.csv"
ISSUE_MAP_PATH = PROJECT_ROOT / "configs/issue_taxonomy_v1.csv"
PAIR_MAP_PATH = PROJECT_ROOT / "configs/valid_product_issue_pairs_v1.csv"

for path in [TAXONOMY_PATH, PRODUCT_MAP_PATH, ISSUE_MAP_PATH, PAIR_MAP_PATH]:
    assert path.exists(), f"No se encontró {path}"

### 1. Contenido del mapa

La regla conservadora solo agrupa cambios de nombre claros. Las demás etiquetas se asignan a sí mismas.

In [13]:
product_map_table = pd.read_csv(PRODUCT_MAP_PATH)
issue_map_table = pd.read_csv(ISSUE_MAP_PATH)
pair_map_table = pd.read_csv(PAIR_MAP_PATH)
product_changes = product_map_table[product_map_table["mapping_reason"] == "clear_rename"]
issue_changes = issue_map_table[issue_map_table["mapping_reason"] == "clear_rename"]

map_summary = pd.DataFrame({
    "dimensión": ["productos", "motivos", "pares válidos originales"],
    "categorías originales": [len(product_map_table), len(issue_map_table), len(pair_map_table)],
    "categorías canónicas": [product_map_table["canonical_product"].nunique(), issue_map_table["canonical_issue"].nunique(), len(pair_map_table)],
    "cambios de nombre propuestos": [len(product_changes), len(issue_changes), 0],
})
display(map_summary)
display(product_changes[["raw_product", "canonical_product"]].reset_index(drop=True))
display(issue_changes[["raw_issue", "canonical_issue"]].reset_index(drop=True))

assert map_summary.loc[0, "categorías originales"] == 21
assert map_summary.loc[0, "categorías canónicas"] == 18
assert map_summary.loc[1, "categorías originales"] == 173
assert map_summary.loc[1, "categorías canónicas"] == 140
assert len(pair_map_table) == 323
assert set(product_map_table["review_status"]) == {TAXONOMY_STATUS}
assert set(issue_map_table["review_status"]) == {TAXONOMY_STATUS}

,dimensión,categorías originales,categorías canónicas,cambios de nombre propuestos
0,productos,21,18,3
1,motivos,173,140,33
2,pares válidos originales,323,323,0


,raw_product,canonical_product
0,Bank account or service,Checking or savings account
1,Credit reporting,Credit reporting or other personal consumer re...
2,"Credit reporting, credit repair services, or o...",Credit reporting or other personal consumer re...


,raw_issue,canonical_issue
0,Adding money,Problem adding money
1,Advertising and marketing,"Advertising and marketing, including promotion..."
2,Can't repay my loan,Struggling to repay your loan
3,Can't stop charges to bank account,Can't stop withdrawals from your bank account
4,Charged bank acct wrong day or amt,Money was taken from your bank account on the ...
5,Charged fees or interest I didn't expect,Charged fees or interest you didn't expect
6,Closing/Cancelling account,Closing your account
7,Cont'd attempts collect debt not owed,Attempts to collect debt not owed
8,Credit monitoring or identity protection,Credit monitoring or identity theft protection...
9,Credit reporting company's investigation,Problem with a company's investigation into an...


### 2. Categorías que no se agrupan

No unimos categorías cuando una etiqueta amplia parece haberse dividido en varias. Ejemplos: `Credit card or prepaid card`; `Consumer Loan`; las categorías históricas de `Money transfers`, `Virtual currency` y `Other financial service`; y las distintas categorías de payday, title, personal y advance loans.

Tampoco unimos motivos parecidos cuando cambian el producto, la etapa o el alcance. Por ejemplo: abrir, cerrar o administrar una cuenta; dificultad para pagar una factura frente a un préstamo; y solicitar un préstamo frente a ser aprobado y no recibir el dinero. Esta decisión evita afirmar equivalencias que los datos públicos no pueden demostrar.

### 3. Archivo resultante

In [14]:
taxonomy_file = pq.ParquetFile(TAXONOMY_PATH)
taxonomy_metadata = taxonomy_file.schema_arrow.metadata
p3_file_summary = pd.DataFrame({
    "archivo": ["normalizado", "con taxonomía"],
    "filas": [normalized_file.metadata.num_rows, taxonomy_file.metadata.num_rows],
    "columnas": [normalized_file.metadata.num_columns, taxonomy_file.metadata.num_columns],
    "tamaño GiB": [NORMALIZED_PATH.stat().st_size / 1024**3, TAXONOMY_PATH.stat().st_size / 1024**3],
})
p3_file_summary["filas"] = p3_file_summary["filas"].map("{:,}".format)
p3_file_summary["tamaño GiB"] = p3_file_summary["tamaño GiB"].map("{:.2f}".format)
display(p3_file_summary)
print(f"Versión: {taxonomy_metadata[b'taxonomy_version'].decode('utf-8')}")
print(f"Estado: {taxonomy_metadata[b'taxonomy_status'].decode('utf-8')}")
print(f"Corte: {taxonomy_metadata[b'taxonomy_freeze_date'].decode('utf-8')}")

assert taxonomy_file.metadata.num_rows == normalized_file.metadata.num_rows == 3_837_184
assert taxonomy_file.metadata.num_columns == normalized_file.metadata.num_columns + 5
assert taxonomy_metadata[b"taxonomy_version"].decode("utf-8") == TAXONOMY_VERSION
assert taxonomy_metadata[b"taxonomy_status"].decode("utf-8") == TAXONOMY_STATUS
assert taxonomy_metadata[b"taxonomy_freeze_date"].decode("utf-8") == FREEZE_DATE.isoformat()

,archivo,filas,columnas,tamaño GiB
0,normalizado,"3,837,184",18,1.70
1,con taxonomía,"3,837,184",23,1.71


Versión: taxonomy_v1_proposed
Estado: proposed_no_business_review
Corte: 2025-06-30


### 4. Conservación de las categorías originales

In [15]:
normalized_category_batches = normalized_file.iter_batches(batch_size=50_000, columns=["Product", "Issue"])
taxonomy_category_batches = taxonomy_file.iter_batches(batch_size=50_000, columns=["Product", "Issue"])
original_categories_are_unchanged = True
for before, after in zip_longest(normalized_category_batches, taxonomy_category_batches):
    if before is None or after is None or not before.equals(after):
        original_categories_are_unchanged = False
        break

display(pd.DataFrame({"comprobación": ["producto y motivo originales sin cambios"], "resultado": [original_categories_are_unchanged]}))
assert original_categories_are_unchanged

,comprobación,resultado
0,producto y motivo originales sin cambios,True


### 5. Categorías posteriores al corte

Solo mostramos cantidades. No usamos los nombres ni actualizamos el mapa con información posterior al corte.

In [16]:
taxonomy_audit = pq.read_table(
    TAXONOMY_PATH,
    columns=["Date received", KNOWN_PRODUCT_COLUMN, KNOWN_ISSUE_COLUMN, KNOWN_PAIR_COLUMN],
)
before_freeze = pc.less_equal(taxonomy_audit["Date received"], pa.scalar(FREEZE_DATE))
after_freeze = pc.invert(before_freeze)

def false_count(column_name, mask):
    values = pc.filter(taxonomy_audit[column_name], mask)
    return len(values) - pc.sum(pc.cast(values, pa.int64())).as_py()

unknown_summary = pd.DataFrame({
    "periodo": ["hasta el corte", "después del corte"],
    "filas": [pc.sum(pc.cast(before_freeze, pa.int64())).as_py(), pc.sum(pc.cast(after_freeze, pa.int64())).as_py()],
    "productos desconocidos": [false_count(KNOWN_PRODUCT_COLUMN, before_freeze), false_count(KNOWN_PRODUCT_COLUMN, after_freeze)],
    "motivos desconocidos": [false_count(KNOWN_ISSUE_COLUMN, before_freeze), false_count(KNOWN_ISSUE_COLUMN, after_freeze)],
    "pares desconocidos": [false_count(KNOWN_PAIR_COLUMN, before_freeze), false_count(KNOWN_PAIR_COLUMN, after_freeze)],
})
display(unknown_summary)

assert unknown_summary.loc[0, "filas"] == 3_203_154
assert unknown_summary.loc[0, ["productos desconocidos", "motivos desconocidos", "pares desconocidos"]].sum() == 0
assert unknown_summary.loc[1, "filas"] == 634_030
assert unknown_summary.loc[1, "productos desconocidos"] == 0
assert unknown_summary.loc[1, "motivos desconocidos"] == 0
assert unknown_summary.loc[1, "pares desconocidos"] == 5

,periodo,filas,productos desconocidos,motivos desconocidos,pares desconocidos
0,hasta el corte,3203154,0,0,0
1,después del corte,634030,0,0,5


## Resultado de P3

- Se conservan las 3,837,184 filas y todas las columnas anteriores.
- Se agregan producto canónico, motivo canónico y tres indicadores de categorías conocidas.
- La propuesta agrupa 21 productos originales en 18 productos canónicos y 173 motivos originales en 140 motivos canónicos.
- Los 323 pares originales observados hasta `2025-06-30` quedan congelados.
- Después del corte no aparecen nombres nuevos de producto o motivo. Sí aparecen 5 filas con una combinación producto–motivo no vista; se marcan como desconocidas y no modifican el mapa.
- `taxonomy_v1_proposed` no representa reglas internas de un banco ni una taxonomía aprobada por negocio.

## P4 — Objetivos y periodos

Un **objetivo** es el resultado histórico que un modelo intentará estimar. `T1` es el motivo canónico; `T2` indica si CFPB registró compensación monetaria o no monetaria; `T3` indica compensación monetaria; y `T4` indica `Timely response? = No`. Estos campos describen el registro CFPB, no decisiones internas de un banco.

Una fila **elegible** es una fila que puede usarse para evaluar un objetivo. La elegibilidad se calcula por separado porque no todos los resultados están disponibles o son comparables en todos los periodos.

In [17]:
import json

from src.data.build_targets import (
    COMPLETE_COLUMNS,
    CONTEXT,
    HOLDOUT,
    KNOWN_T1_COLUMN,
    NO_SHARED_COLUMNS,
    OOD,
    PERIOD_COLUMN,
    TARGETS_VERSION,
    TRAIN,
    VALIDATION,
)

TARGETS_PATH = PROJECT_ROOT / "data/interim/targets_periods.parquet"
HOLDOUT_STATUS_PATH = PROJECT_ROOT / "configs/holdout_review_status.json"

for path in [TARGETS_PATH, HOLDOUT_STATUS_PATH]:
    assert path.exists(), f"No se encontró {path}; ejecuta dvc repro"

period_order = [CONTEXT, TRAIN, VALIDATION, HOLDOUT, OOD]
complete_columns = list(COMPLETE_COLUMNS.values())
no_shared_columns = list(NO_SHARED_COLUMNS.values())

### 1. Archivo y periodos

Los periodos respetan el orden del tiempo: contexto histórico, aprendizaje, validación posterior, reserva 2025-H2 y datos parciales de 2026.

In [18]:
targets_file = pq.ParquetFile(TARGETS_PATH)
targets_metadata = targets_file.schema_arrow.metadata
audit_columns = [
    PERIOD_COLUMN,
    KNOWN_T1_COLUMN,
    "T2",
    "T3",
    "T4",
    *complete_columns,
    *no_shared_columns,
]
targets_audit = pq.read_table(TARGETS_PATH, columns=audit_columns).to_pandas()
period_summary = (
    targets_audit.groupby(PERIOD_COLUMN, observed=True)
    .size()
    .reindex(period_order)
    .rename("filas")
    .to_frame()
)
display(period_summary)
print(f"Filas: {targets_file.metadata.num_rows:,}")
print(f"Columnas: {targets_file.metadata.num_columns}")
print(f"Tamaño: {TARGETS_PATH.stat().st_size / 1024**3:.2f} GiB")
print(f"Versión: {targets_metadata[b'targets_periods_version'].decode('utf-8')}")

expected_period_rows = [1_206_176, 1_301_794, 695_184, 526_872, 107_158]
assert period_summary["filas"].tolist() == expected_period_rows
assert targets_file.metadata.num_rows == 3_837_184
assert targets_file.metadata.num_columns == 44
assert targets_metadata[b"targets_periods_version"].decode("utf-8") == TARGETS_VERSION

,filas
period,
context_2015_2022,1206176
train_2023_2024,1301794
validation_2025_h1,695184
holdout_2025_h2,526872
ood_2026_partial,107158


Filas: 3,837,184
Columnas: 44
Tamaño: 1.71 GiB
Versión: targets_periods_v1


### 2. Resultados conocidos y desconocidos

`T1` tiene varias categorías, por eso no se divide en positivo y negativo. `T2`, `T3` y `T4` son objetivos binarios: **positivo** significa que el evento está registrado y **negativo** que no lo está según la definición acordada. **Desconocido** significa que el valor original no permite asignar ninguna de esas dos opciones.

In [19]:
target_summary = pd.DataFrame([
    {
        "objetivo": "T1 — motivo canónico",
        "conocidos": int(targets_audit[KNOWN_T1_COLUMN].sum()),
        "positivos": "No aplica",
        "negativos": "No aplica",
        "desconocidos": int((~targets_audit[KNOWN_T1_COLUMN]).sum()),
    },
    *[
        {
            "objetivo": target,
            "conocidos": int(targets_audit[target].notna().sum()),
            "positivos": int(targets_audit[target].eq(True).sum()),
            "negativos": int(targets_audit[target].eq(False).sum()),
            "desconocidos": int(targets_audit[target].isna().sum()),
        }
        for target in ["T2", "T3", "T4"]
    ],
])
display(target_summary.set_index("objetivo"))

assert target_summary.loc[0, "desconocidos"] == 5
assert target_summary.loc[1, "positivos"] == 1_273_365
assert target_summary.loc[2, "positivos"] == 98_150
assert target_summary.loc[3, "positivos"] == 42_615

,conocidos,positivos,negativos,desconocidos
objetivo,,,,
T1 — motivo canónico,3837179,No aplica,No aplica,5
T2,3822231,1273365,2548866,14953
T3,3822231,98150,3724081,14953
T4,3837184,42615,3794569,0


### 3. Dos vistas de evaluación

La vista **completa** conserva todas las filas cuyo objetivo es conocido. La vista **sin texto compartido** es más estricta: también excluye narrativas cuyo hash ya apareció en un periodo usado como referencia. Así se evita evaluar como nuevo un texto que el modelo pudo haber visto antes. Los duplicados dentro del mismo periodo se conservan.

In [20]:
complete_summary = (
    targets_audit.groupby(PERIOD_COLUMN, observed=True)[complete_columns]
    .sum()
    .reindex(period_order)
    .astype("int64")
)
complete_summary.columns = ["T1", "T2", "T3", "T4"]

no_shared_summary = (
    targets_audit.groupby(PERIOD_COLUMN, observed=True)[no_shared_columns]
    .sum()
    .reindex(period_order)
    .astype("int64")
)
no_shared_summary.columns = ["T1", "T2", "T3", "T4"]

print("Vista completa")
display(complete_summary)
print("Vista sin texto compartido con periodos anteriores")
display(no_shared_summary)

holdout_status = json.loads(HOLDOUT_STATUS_PATH.read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "estado 2025-H2": holdout_status["status"],
    "IDs esperados": holdout_status["expected_reviewed_ids"],
    "IDs recuperados": holdout_status["recovered_reviewed_ids"],
    "evaluación final permitida": holdout_status["final_evaluation_allowed"],
}]))

assert complete_summary.loc[HOLDOUT].sum() == 0
assert no_shared_summary.loc[HOLDOUT].sum() == 0
assert complete_summary.loc[OOD, ["T2", "T3", "T4"]].sum() == 0
assert complete_summary.loc[OOD, "T1"] == 107_155
assert holdout_status["final_evaluation_allowed"] is False

Vista completa


,T1,T2,T3,T4
period,,,,
context_2015_2022,1206176,1198659,1198659,1206176
train_2023_2024,1301794,1300650,1300650,1301794
validation_2025_h1,695184,693502,693502,695184
holdout_2025_h2,0,0,0,0
ood_2026_partial,107155,0,0,0


Vista sin texto compartido con periodos anteriores


,T1,T2,T3,T4
period,,,,
context_2015_2022,1206176,1198659,1198659,1206176
train_2023_2024,1301794,1300650,1300650,1301794
validation_2025_h1,564813,563148,563148,564813
holdout_2025_h2,0,0,0,0
ood_2026_partial,102557,0,0,0


,estado 2025-H2,IDs esperados,IDs recuperados,evaluación final permitida
0,ids_not_recovered,25000,0,False


## Resultado de P4

- Se conservan las 3,837,184 filas y las 23 columnas anteriores; se agregan 21 columnas de objetivos, periodos y elegibilidad.
- Los 5 `T1` desconocidos corresponden a combinaciones producto–motivo no vistas antes del corte. No mostramos sus nombres ni actualizamos la taxonomía con periodos posteriores.
- `2025-H2` tiene objetivos observados, pero cero filas elegibles. Los 25,000 IDs revisados anteriormente no se recuperaron, por lo que no podemos separar de forma verificable esos casos ni sus grupos de texto. El periodo queda bloqueado y no es una evaluación final intacta.
- En `2026_partial`, `T1` puede evaluarse. `T2`–`T4` quedan bloqueados hasta definir la **maduración**: cuánto tiempo debe pasar para considerar que sus resultados ya están completos.
- La vista sin texto compartido deja 564,813 filas para validar `T1`, frente a 695,184 en la vista completa. Esta diferencia muestra por qué una evaluación que ignore textos repetidos sería demasiado optimista.

## P5 — Tabla preparada para modelado

P5 no transforma los datos. Valida el contrato de P4 y copia el archivo sin cambiar sus bytes a `data/processed/prepared.parquet`, que será la entrada de `Modeling/pipaber`.

In [21]:
from src.data.finalize_prepared import (
    EXPECTED_METADATA,
    EXPECTED_PREPARED_COLUMNS,
    EXPECTED_ROW_COUNT,
)

PREPARED_PATH = PROJECT_ROOT / "data/processed/prepared.parquet"
assert PREPARED_PATH.exists(), f"No se encontró {PREPARED_PATH}; ejecuta dvc repro"

prepared_file = pq.ParquetFile(PREPARED_PATH)
prepared_metadata = prepared_file.schema_arrow.metadata or {}
prepared_contract = pq.read_table(
    PREPARED_PATH,
    columns=["Complaint ID", PERIOD_COLUMN],
)
prepared_ids = prepared_contract["Complaint ID"]
prepared_periods = prepared_contract[PERIOD_COLUMN]

prepared_summary = pd.DataFrame([{
    "filas": prepared_file.metadata.num_rows,
    "columnas": prepared_file.metadata.num_columns,
    "tamaño GiB": round(PREPARED_PATH.stat().st_size / 1024**3, 2),
    "IDs ausentes": prepared_ids.null_count,
    "IDs distintos": pc.count_distinct(prepared_ids).as_py(),
    "periodos": len(pc.unique(prepared_periods)),
}])
display(prepared_summary)
print(f"Normalizador: {prepared_metadata[b'text_normalizer'].decode('utf-8')}")
print(f"Taxonomía: {prepared_metadata[b'taxonomy_version'].decode('utf-8')}")
print(f"Estado de taxonomía: {prepared_metadata[b'taxonomy_status'].decode('utf-8')}")
print(f"Objetivos y periodos: {prepared_metadata[b'targets_periods_version'].decode('utf-8')}")

assert prepared_file.metadata.num_rows == EXPECTED_ROW_COUNT
assert prepared_file.schema_arrow.names == EXPECTED_PREPARED_COLUMNS
assert prepared_ids.null_count == 0
assert pc.count_distinct(prepared_ids).as_py() == EXPECTED_ROW_COUNT
assert set(pc.unique(prepared_periods).to_pylist()) == set(period_order)
assert prepared_file.schema_arrow == targets_file.schema_arrow
assert PREPARED_PATH.stat().st_size == TARGETS_PATH.stat().st_size
for key, expected_value in EXPECTED_METADATA.items():
    assert prepared_metadata[key] == expected_value

,filas,columnas,tamaño GiB,IDs ausentes,IDs distintos,periodos
0,3837184,44,1.71,0,3837184,5


Normalizador: text_normalizer_v1
Taxonomía: taxonomy_v1_proposed
Estado de taxonomía: proposed_no_business_review
Objetivos y periodos: targets_periods_v1


## Resultado de P5

- La entrega final contiene 3,837,184 filas, 44 columnas y 3,837,184 IDs únicos.
- Las versiones del normalizador, la taxonomía y los objetivos quedan registradas dentro del Parquet.
- La tabla final es una copia de P4, no una transformación adicional. DVC registra el mismo hash para ambos archivos.
- Limitaciones que viajan con la entrega: la taxonomía sigue siendo una propuesta técnica; `2025-H2` está bloqueado; `T2`–`T4` de 2026 no están maduros; y los datos CFPB no representan reglas ni resultados internos de un banco.
- Archivo entregado a la rama de modelado: `data/processed/prepared.parquet`. Esta rama no incluye TF-IDF, BGE, FAISS ni modelos.